# Using Lacuna with Apptainer

This guide shows how to run Lacuna analyses using Apptainer (formerly Singularity) containers, commonly used on HPC clusters where Docker is not available.

**What you'll learn:**

- Convert the Docker image to an Apptainer SIF file
- Bind host directories into the container
- Run a structural network mapping analysis with Apptainer


## Prerequisites

You need Apptainer installed on your system. See [Installing Apptainer](https://apptainer.org/docs/admin/main/installation.html#optional-requirements) for installation instructions. Check with:

```bash
apptainer --version
```

On HPC clusters, Apptainer is typically available as a module:

```bash
module load apptainer  # or singularity
```

## Setup

Pull the Lacuna container image. Apptainer can directly convert Docker images to SIF format.

In [ ]:
!apptainer pull /tmp/lacuna.sif docker://ghcr.io/m-petersen/lacuna:edge

This creates a `lacuna_edge.sif` file. Verify it works.

In [10]:
!apptainer run /tmp/lacuna.sif --help

usage: lacuna [-h] [--version] <command> ...

Lacuna v0.0.1.dev385

options:
  -h, --help  show this help message and exit
  --version   show program's version number and exit

commands:
  Use 'lacuna <command> --help' for more information.

  <command>
    fetch     Download and setup connectomes
    run       Run lesion network mapping analyses
    collect   Aggregate parcelstats across subjects
    info      Display available resources (atlases, connectomes)
    bidsify   Convert NIfTI files to BIDS format
    tutorial  Setup tutorial data for learning Lacuna

Commands:
  bidsify   Convert NIfTI files to BIDS format
  fetch     Download and setup connectomes for analysis
  run       Run lesion analyses
  collect   Aggregate results across subjects
  info      Display available resources (atlases, connectomes)
  tutorial  Setup tutorial data for learning Lacuna

Examples:
  lacuna tutorial ./my_tutorial
  lacuna fetch gsp1000 --api-key \$DATAVERSE_API_KEY
  lacuna run rd /bids /outpu

## How bind paths work

Apptainer requires explicit bind mounts for paths outside the container using `--bind`:

```
--bind /host/path:/container/path:ro   # read-only
--bind /host/path:/container/path      # read-write
```

| Purpose | Host path | Container path | Mode |
|---------|-----------|----------------|------|
| BIDS input data | `/path/to/bids` | `/bids` | read-only (`:ro`) |
| Output results | `/path/to/output` | `/output` | read-write |
| Connectomes | `/path/to/connectomes` | `/connectomes` | read-only (`:ro`) |

Apptainer automatically binds `$HOME`, `$PWD`, and `/tmp`.

The container entrypoint is the `lacuna` command, so you pass subcommands directly after the SIF file.

## Prepare tutorial data

Use Lacuna inside the container to create a tutorial dataset.

In [11]:
!mkdir -p /tmp/apptainer_tutorial/
!apptainer run \
    --bind /tmp/apptainer_tutorial:/data \
    /tmp/lacuna.sif \
    tutorial /data/bids --force


Setting up tutorial data at: /data/bids
✓ Tutorial data copied to: /data/bids

The tutorial dataset includes:
  - 3 synthetic subjects (sub-01, sub-02, sub-03)
  - Binary lesion masks in MNI152NLin6Asym space
  - BIDS-compliant structure


In [12]:
!ls /tmp/apptainer_tutorial/bids/

dataset_description.json  participants.tsv  sub-01  sub-03
__init__.py		  __pycache__	    sub-02


Fetch the HCP1065 tractogram and run SNM on a single subject.

In [13]:
!apptainer run \
    --bind /tmp/apptainer_tutorial:/data \
    /tmp/lacuna.sif \
    fetch hcp1065 --output-dir /data/connectomes

Fetching HCP1065 structural tractogram...
  Output: /data/connectomes
  Keep original: True

hcp1065_avg_tracts_trk.zip: 100%|████████████| 588M/588M [00:43<00:00, 13.5MB/s]
Found 77 tract files (10 excluded)
Loading and merging streamlines...
Merging tracts: 100%|███████████████████████████| 77/77 [00:15<00:00,  4.87it/s]
Processed 77 files, 479804 total streamlines
Saving merged tractogram to /data/connectomes/hcp1065.tck...
Merge complete: /data/connectomes/hcp1065.tck

✓ HCP1065 fetch complete!
  Files: 2
  Duration: 67.0s
  Output: /data/connectomes


In [14]:
!ls /tmp/apptainer_tutorial/connectomes/

hcp1065_avg_tracts_trk.zip  hcp1065.tck  hcp1065_tracts


## Run structural network mapping

In [19]:
!mkdir -p /tmp/apptainer_tutorial/output
!apptainer run \
    --bind /tmp/apptainer_tutorial/bids:/bids:ro \
    --bind /tmp/apptainer_tutorial/connectomes:/connectomes:ro \
    --bind /tmp/apptainer_tutorial/output:/output \
    /tmp/lacuna.sif \
    run snm /bids /output \
    --connectome-path /connectomes/hcp1065.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym

Inspect the output. The results are written to the host filesystem via the bind mount.

In [18]:
!ls /tmp/apptainer_tutorial/output/sub-01/ses-01/anat/

sub-01_ses-01_desc-provenance.json
sub-01_ses-01_label-acuteinfarct_summarystatistics_stats.json
sub-01_ses-01_space-MNI152NLin2009cAsym_label-acuteinfarct_mask.nii.gz
sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_method-snm_desc-disconnectionpct.json
sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_method-snm_desc-disconnectionpct.nii.gz
